# Chapter 4: Strategic Form Games, with Gambit

This notebook accompanies Chapter 4 of *An Introduction to Computational Game Theory*.
It recomputes the chapter's examples with [Gambit](https://www.gambit-project.org/)
(the `pygambit` Python package, version 17) and, where the chapter describes an algorithm,
gives a short Python implementation next to Gambit's own.

Section headings follow the chapter. Cells marked **Beyond the book** go a little
further than the text.

**Setup.** `pip install pygambit numpy pandas matplotlib`. The helper module `cgtbook.py`
(in this folder) only builds games with the book's strategy names and prints results in
the book's notation; everything game-theoretic is done either by Gambit or in this notebook.

In [ ]:
import itertools
from fractions import Fraction

import numpy as np
import pandas as pd
from cgtbook import bimatrix, nice, payoff_arrays, pure_profile_names, table_game

import pygambit as gbt

print("pygambit", gbt.__version__)

## 4.1 Game forms and games

### Payoff matrices

The chapter's running example has player 1 (rows) choosing $T$ or $B$ and player 2
(columns) choosing $l$ or $r$. As in the book, the first matrix holds player 1's payoffs
and the second player 2's, with rows indexed by player 1's strategies.

In [ ]:
A = [[1, 1],
     [0, 0]]
B = [[1, 0],
     [0, 1]]
running = bimatrix(A, B, ["T", "B"], ["l", "r"], title="Running example (Chapter 4)")
running

Gambit stores a strategic form game as a table of outcomes. Players and strategies are
referred to by their labels, and `to_arrays` gives back one payoff array per player,
indexed by pure strategy profiles.

In [ ]:
print(list(running.players), {p: list(running.get_strategies(p)) for p in running.players})
running.to_arrays()

## 4.2 Solution concepts

### The concept of "best response"

For a pure strategy profile $\vec\sigma$, $BR_i(\vec\sigma)$ is the set of player $i$'s
strategies that maximise $u_i(\vec\sigma_{-i}, \sigma_i')$. The function below works for any
number of players, directly from the payoff arrays.

In [ ]:
def best_responses(g, profile):
    """BR_i(profile) for every player i; profile is a tuple of strategy indices."""
    U = payoff_arrays(g)
    names = [list(g.get_strategies(p)) for p in g.players]
    br = {}
    for i, player in enumerate(g.players):
        # player i's payoff from each strategy s, the others playing as in profile
        values = [U[i][profile[:i] + (s,) + profile[i + 1:]] for s in range(len(names[i]))]
        best = max(values)
        br[player] = {names[i][s] for s, v in enumerate(values) if v == best}
    return br


def br_table(g):
    names = [list(g.get_strategies(p)) for p in g.players]
    rows = []
    for prof in itertools.product(*(range(len(n)) for n in names)):
        br = best_responses(g, prof)
        row = {"profile": tuple(names[i][s] for i, s in enumerate(prof))}
        row.update({f"BR_{p}": "{" + ", ".join(sorted(b)) + "}" for p, b in br.items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index("profile")


br_table(running)

As in the book: $BR_1 = \{T\}$ whatever player 2 does, while player 2's best response
copies player 1 ($l$ against $T$, $r$ against $B$).

### Dominant strategies

The chapter's algorithm for dominant strategies (a strategy is dominant if it is a best
response to every profile) intersects the best response sets over all profiles.

In [ ]:
def dominant_strategies(g, player):
    names = [list(g.get_strategies(p)) for p in g.players]
    ds = set(g.get_strategies(player))
    for prof in itertools.product(*(range(len(n)) for n in names)):
        ds &= best_responses(g, prof)[player]
    return ds or None


{p: dominant_strategies(running, p) for p in running.players}

$T$ is dominant for player 1; player 2 has no dominant strategy.

### Nash equilibrium

`gbt.nash.enumpure_solve` enumerates all pure strategy Nash equilibria. It returns a
result object; each equilibrium is a (degenerate) mixed strategy profile.

In [ ]:
result = gbt.nash.enumpure_solve(running)
[pure_profile_names(eq) for eq in result.equilibria]

$(T, l)$ is the unique pure strategy Nash equilibrium. Any profile, equilibrium or not,
can be inspected through its payoffs and *regrets* (how much a player could gain by
deviating). A profile is a Nash equilibrium exactly when every regret is zero.

In [ ]:
for eq in result.equilibria:
    print(pure_profile_names(eq), "payoffs", nice(eq.payoffs), "max regret", eq.max_regret())

not_eq = running.mixed_strategy_profile([[1, 0], [0, 1]], rational=True)   # (T, r)
print("(T, r): payoffs", nice(not_eq.payoffs), " regrets", nice(not_eq.strategy_regrets))

At $(T,r)$ player 2 regrets not playing $l$, so $(T,r)$ is not an equilibrium, as the
book observes.

### Eliminating dominated strategies

The worked example of iterated elimination of strictly dominated strategies (IEDS):

In [ ]:
ieds_game = bimatrix(
    [[4, 5, 6], [2, 8, 3], [3, 9, 2]],
    [[3, 1, 2], [1, 4, 6], [0, 6, 8]],
    ["T", "M", "B"], ["l", "c", "r"], title="IEDS example",
)
ieds_game

`gbt.supports.undominated_strategies_solve` performs one round of elimination for all
players at once, returning a *support profile* (a subset of each player's strategies).
With `strict=True` only strictly dominated strategies are removed, as in the book; like
the book, Gambit only considers domination by pure strategies.

In [ ]:
support = ieds_game.strategy_support_profile()
round_ = 0
while True:
    print(f"round {round_}:", {p: list(support[p]) for p in ieds_game.players})
    new = gbt.supports.undominated_strategies_solve(support, strict=True)
    if all(list(new[p]) == list(support[p]) for p in ieds_game.players):
        break
    support, round_ = new, round_ + 1

The order matches the book: first $c$ goes (dominated by $r$), then $M$ and $B$ (dominated
by $T$), then $r$ (dominated by $l$), leaving $(T, l)$. The game is dominance solvable,
so $(T,l)$ is its unique pure Nash equilibrium:

In [ ]:
[pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(ieds_game).equilibria]

## 4.3 Social welfare and 4.4 Some important games

The chapter's gallery of $2\times 2$ games (macros in `example-games.tex`). For each we
list the pure Nash equilibria (from Gambit), the profiles that maximise utilitarian and
egalitarian welfare, and the Pareto optimal profiles.

In [ ]:
gallery = {
    "Prisoner's Dilemma": (
        [[-2, 0], [-3, -1]], [[-2, -3], [0, -1]], ["D", "C"], ["d", "c"]),
    "Prisoner's Dilemma (Axelrod)": (
        [[1, 5], [0, 3]], [[1, 0], [5, 3]], ["D", "C"], ["d", "c"]),
    "Coordination": (
        [[1, 0], [0, 1]], [[1, 0], [0, 1]], ["A", "B"], ["a", "b"]),
    "Payoff-dominant coordination": (
        [[100, 0], [0, 1]], [[100, 0], [0, 1]], ["T", "B"], ["l", "r"]),
    "Bach or Stravinsky": (
        [[2, 0], [0, 1]], [[1, 0], [0, 2]], ["Bach", "Stravinsky"], ["bach", "stravinsky"]),
    "Chicken": (
        [[0, 4], [2, 3]], [[0, 2], [4, 3]], ["D", "C"], ["d", "c"]),
    "Stag Hunt": (
        [[7, 0], [4, 4]], [[7, 4], [0, 4]], ["Deer", "Hare"], ["deer", "hare"]),
    "Matching Pennies": (
        [[1, -1], [-1, 1]], [[-1, 1], [1, -1]], ["Heads", "Tails"], ["heads", "tails"]),
}
games = {name: bimatrix(A, B, r, c, title=name) for name, (A, B, r, c) in gallery.items()}


def outcomes(g):
    """All pure profiles with their payoff vectors."""
    U = payoff_arrays(g)
    names = [list(g.get_strategies(p)) for p in g.players]
    for prof in itertools.product(*(range(len(n)) for n in names)):
        yield tuple(names[i][s] for i, s in enumerate(prof)), tuple(u[prof] for u in U)


def pareto_optimal(g):
    outs = list(outcomes(g))

    def dominated(u):
        return any(all(a >= b for a, b in zip(v, u, strict=False)) and v != u for _, v in outs)
    return [p for p, u in outs if not dominated(u)]


def welfare_summary(g):
    outs = list(outcomes(g))
    ne = [pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(g).equilibria]
    sw = {p: sum(u) for p, u in outs}
    esw = {p: min(u) for p, u in outs}

    def show(profs):
        return " ".join("(" + ",".join(p) + ")" for p in profs) or "none"
    return {
        "pure NE": show(ne),
        "max sw": show(p for p in sw if sw[p] == max(sw.values())),
        "max esw": show(p for p in esw if esw[p] == max(esw.values())),
        "Pareto optimal": show(pareto_optimal(g)),
    }


pd.DataFrame({name: welfare_summary(g) for name, g in games.items()}).T

Matching Pennies has no pure equilibrium; its mixed equilibrium is the starting point of
Chapter 5.

## 4.5 Revisiting the Price of Anarchy

$\mathit{PoA}(G) = \max_\omega sw(\omega) \,/\, \min_{\vec\sigma \in NE(G)} sw(\vec\sigma)$,
over pure equilibria.

In [ ]:
def price_of_anarchy(g):
    ne = gbt.nash.enumpure_solve(g).equilibria
    if not ne:
        return None
    best = max(sum(u) for _, u in outcomes(g))
    worst_ne = min(sum(eq.payoffs[p] for p in g.players) for eq in ne)
    return str(Fraction(best) / Fraction(str(worst_ne)))


{name: price_of_anarchy(games[name]) for name in
 ["Prisoner's Dilemma (Axelrod)", "Chicken", "Coordination", "Bach or Stravinsky",
  "Stag Hunt", "Matching Pennies"]}

These are the values in the book: 3 for the Axelrod Prisoner's Dilemma, 1 for Chicken,
Coordination and Bach or Stravinsky, $14/8 = 7/4$ for the Stag Hunt, and undefined for
Matching Pennies.

## 4.6 Computing pure strategy Nash equilibria

### Exhaustive search

The chapter's first algorithm checks every profile against every unilateral deviation;
the improved version computes, for each player and each $\vec\sigma_{-i}$, the best payoff
once and marks the profiles that do not achieve it. Both work for any number of players.

In [ ]:
def exhaustive_search(g):
    U = payoff_arrays(g)
    sizes = U[0].shape
    ne = []
    for prof in itertools.product(*(range(s) for s in sizes)):
        stable = True
        for i in range(len(sizes)):
            for s in range(sizes[i]):
                dev = prof[:i] + (s,) + prof[i + 1:]
                if U[i][dev] > U[i][prof]:
                    stable = False
        if stable:
            ne.append(prof)
    return ne


def improved_search(g):
    U = payoff_arrays(g)
    marked = np.zeros(U[0].shape, dtype=bool)
    for i, u in enumerate(U):
        best = u.max(axis=i, keepdims=True)   # best payoff for each sigma_{-i}
        marked |= (u < best)
    return [tuple(int(k) for k in idx) for idx in np.argwhere(~marked)]


for _name, g in games.items():
    names = [list(g.get_strategies(p)) for p in g.players]

    def as_names(profs, names=names):
        return [tuple(names[i][s] for i, s in enumerate(p)) for p in profs]

    gambit = sorted(pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(g).equilibria)
    assert sorted(as_names(exhaustive_search(g))) == sorted(as_names(improved_search(g))) == gambit
print("exhaustive search, improved search and enumpure agree on all games in the gallery")

### Gambit supports any number of players

The chapter's public goods game has continuous contributions, so it is not a finite game.
A finite version: each of $n$ players either contributes their endowment $e = 1$ or
nothing, and $u_i = e - c_i + \frac{\alpha}{n}\sum_j c_j$ with $1 < \alpha < n$.
Contributing nothing is strictly dominant, so the unique Nash equilibrium is that nobody
contributes, although everybody would be better off if all contributed.

In [ ]:
def public_goods(n, alpha=Fraction(3, 2)):
    shape = (2,) * n                      # strategy 0 = "keep", 1 = "give"
    U = [np.empty(shape, dtype=object) for _ in range(n)]
    for prof in itertools.product((0, 1), repeat=n):
        total = sum(prof)
        for i in range(n):
            U[i][prof] = 1 - prof[i] + alpha * Fraction(total, n)
    return table_game(U, [["keep", "give"]] * n, title=f"Public goods, n={n}")


pg3 = public_goods(3)
print([pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(pg3).equilibria])
undominated = gbt.supports.undominated_strategies_solve(pg3, strict=True)
print({p: list(undominated[p]) for p in pg3.players})

The size of the explicit representation grows exponentially with $n$: the table has
$n \cdot 2^n$ payoffs. Exhaustive search is polynomial in this input size, but the input
itself is exponential in $n$, which is the chapter's point about explicit versus succinct
representations.

In [ ]:
import time

rows = []
for n in range(2, 13, 2):
    g = public_goods(n)
    t0 = time.perf_counter()
    ne = gbt.nash.enumpure_solve(g).equilibria
    rows.append({"n": n, "payoff entries n*2^n": n * 2**n, "pure NE": len(ne),
                 "enumpure time (s)": round(time.perf_counter() - t0, 4)})
pd.DataFrame(rows).set_index("n")

### Myopic best response

Starting from some profile, players in turn switch to a best response if they have a
strictly better reply. The chapter's example shows that this can cycle for ever.

In [ ]:
mbr_game = bimatrix(
    [[-1, 1, -2], [1, -1, -2], [-2, -2, 2]],
    [[1, -1, -2], [-1, 1, -2], [-2, -2, 2]],
    ["T", "M", "B"], ["l", "c", "r"], title="Myopic best response cycles",
)
mbr_game

In [ ]:
def myopic_best_response(g, start, max_rounds=10):
    """Return the sequence of profiles visited (as strategy names)."""
    U = payoff_arrays(g)
    names = [list(g.get_strategies(p)) for p in g.players]
    prof = tuple(names[i].index(s) for i, s in enumerate(start))
    path = [prof]
    for _ in range(max_rounds):
        changed = False
        for i in range(len(names)):
            payoffs = [U[i][prof[:i] + (s,) + prof[i + 1:]] for s in range(len(names[i]))]
            if max(payoffs) > U[i][prof]:
                prof = prof[:i] + (payoffs.index(max(payoffs)),) + prof[i + 1:]
                path.append(prof)
                changed = True
        if not changed:
            break
    return [tuple(names[i][s] for i, s in enumerate(p)) for p in path]


" -> ".join("(" + ",".join(p) + ")" for p in myopic_best_response(mbr_game, ("T", "l"), 3))

The dynamics circle around the top-left $2\times 2$ block (which is Matching Pennies) and
never reach the unique pure equilibrium, which Gambit finds immediately:

In [ ]:
[pure_profile_names(eq) for eq in gbt.nash.enumpure_solve(mbr_game).equilibria]

Looking ahead to Chapters 5 and 6: this game also has two mixed equilibria, which
`enummixed_solve` (Chapter 6) finds.

In [ ]:
from cgtbook import eq_table

eq_table(gbt.nash.enummixed_solve(mbr_game).equilibria)

## 4.7 Potential games (Beyond the book)

The chapter proves that myopic best response always terminates in a potential game but
leaves concrete examples to the chapter on compact games. A tiny congestion game: three
drivers each choose road $X$ or $Y$; road $X$ costs the number of drivers using it, road
$Y$ always costs $5/2$. Payoffs are minus costs. Rosenthal's potential shows this is a
potential game, so myopic best response must stop, wherever it starts. The equilibria are
exactly the profiles with two drivers on road $X$.

In [ ]:
U = [np.empty((2, 2, 2), dtype=object) for _ in range(3)]
for prof in itertools.product((0, 1), repeat=3):
    on_x = prof.count(0)
    for i in range(3):
        U[i][prof] = -on_x if prof[i] == 0 else Fraction(-5, 2)
congestion = table_game(U, [["X", "Y"]] * 3, title="Three-driver congestion game")

ne = gbt.nash.enumpure_solve(congestion).equilibria
print("pure NE:", [pure_profile_names(eq) for eq in ne])
print("myopic best response from every starting profile:")
for start in itertools.product("XY", repeat=3):
    path = myopic_best_response(congestion, start)
    print(" -> ".join("".join(p) for p in path))